# Setup

## Install library

In [ ]:
!pip install sastrawi
!pip install swifter

## Import library

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import seaborn as sns
import swifter
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory, StopWordRemover, ArrayDictionary
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

import shutil
from google.colab import drive, files

# Load Dataset & Exploratory Data Analysis

## Import dataset from GitHub

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/tantowjy/news-classification/refs/heads/main/dataset/hoax/dataset-final.csv")

## Exploratory Data Analysis

### Class distribution

In [ ]:
# count the occurrences of each class in 'is_fake'
count_classes = df['is_fake'].value_counts()

# plot the distribution as a bar chart
plt.figure(figsize=(8, 6))
count_classes.plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Distribution of Fake and Factual News')
plt.xlabel('is_fake')
plt.ylabel('count')
plt.xticks(ticks=[0, 1], labels=['factual (0)', 'fake (1)'])
plt.show()

### Missing values

In [ ]:
df.isna().sum()

### Median and mean word count


In [ ]:
# add a column for the word count
df['text'] = df['title'] + ' ' + df['content']
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

# calculate the median word count for each label
median_word_count_label_0 = df[df['is_fake'] == 0]['word_count'].median()
median_word_count_label_1 = df[df['is_fake'] == 1]['word_count'].median()

median_word_count_label_0, median_word_count_label_1

In [ ]:
# calculate the median word count for each label
mean_word_count_label_0 = df[df['is_fake'] == 0]['word_count'].mean()
mean_word_count_label_1 = df[df['is_fake'] == 1]['word_count'].mean()

mean_word_count_label_0, mean_word_count_label_1

In [ ]:
# plot word count
fig, (fact, fake) = plt.subplots(1, 2, figsize=(10, 4))

fact_words = df[df['is_fake'] == 1]['word_count']
fake_words = df[df['is_fake'] == 0]['word_count']

fact.hist(fact_words, color='skyblue')
fake.hist(fake_words, color='salmon')

fact.set_title('Factual News')
fake.set_title('Fake News')

fig.suptitle('Words per news articles')
plt.show()

# Data Preprocessing

## Stopword removal and lemmatization

In [ ]:
# create sastrawi stopword
stopword_factory = StopWordRemoverFactory()
stopword = stopword_factory.create_stop_word_remover()

# create sastrawi stemmer
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

In [ ]:
# functions for cleaning, removing stopwords
def preprocess_text(text):
    text = str(text)

    # change text to lowercase
    text = text.lower()

    # change link with http/https patterns
    text = re.sub(r'http\S+', '', text)

    # remove hashtag and username
    text = re.sub(r'(@\w+|#\w+)', '', text)

    # remove character other than a-z and A-Z
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # replace new line '\n' with space
    text = re.sub(r'\n', ' ', text)

    # remove stopword with sastrawi library
    text = stopword.remove(text)

    # do stemming with sastrawi library
    text = stemmer.stem(text)

    # removing more than one space
    text = re.sub(r'\s{2,}', ' ', text)

    return text

In [ ]:
# dataframe information
df.info()

In [ ]:
# text preprocessing
df['text'] = df['text'].swifter.apply(preprocess_text)

## Save text preprocessing result

In [ ]:
# save data preprocessing result
df.to_csv("dataset-final-preprocess.csv", index=False)
df.to_json("dataset-final-preprocess.json")

In [ ]:
files.download("dataset-final-preprocess.csv")
files.download("dataset-final-preprocess.json")

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/tantowjy/news-classification/refs/heads/main/dataset/hoax/dataset-final-preprocess.csv")

## Check median and mean word count after preprocessing

In [ ]:
# add a column for the word count
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

# calculate the median word count for each label
median_word_count_label_0 = df[df['is_fake'] == 0]['word_count'].median()
median_word_count_label_1 = df[df['is_fake'] == 1]['word_count'].median()

median_word_count_label_0, median_word_count_label_1

In [ ]:
# calculate the median word count for each label
mean_word_count_label_0 = df[df['is_fake'] == 0]['word_count'].mean()
mean_word_count_label_1 = df[df['is_fake'] == 1]['word_count'].mean()

mean_word_count_label_0, mean_word_count_label_1

## Split the datasets

In [ ]:
# delete null row
df = df.dropna()

# separating features and labels
X = df['text'].values
y = df['is_fake'].values

In [ ]:
# Split the dataset into training and testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Baseline Model with ScikitLearn

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# initialize classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=10000),
    "Support Vector Machine": SVC(),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
}

# create a pipeline for each classifier
pipelines = {name: Pipeline([('tfidf', TfidfVectorizer()), ('clf', clf)]) for name, clf in classifiers.items()}

# train and evaluate classifiers
n_classifiers = len(pipelines)
n_cols = 2
n_rows = (n_classifiers + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 6*n_rows))
axes = axes.flatten()

for idx, (name, pipeline) in enumerate(pipelines.items()):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    # calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name}: {accuracy:.2f}")

    # compute confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    # plot confusion matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Factual', 'Fake'], yticklabels=['Factual', 'Fake'], ax=axes[idx])
    axes[idx].set_title(f"Confusion Matrix: {name}")
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

# hide any remaining empty subplots
for j in range(idx + 1, n_rows * n_cols):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## Create Tokenizer

In [ ]:
vocab_size = 10000
embedding_dim = 16
max_length = 100
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"

In [ ]:
# Create tokenizer
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)

# Converting text to numeric squences
train_sequences = tokenizer.texts_to_sequences(X_train)
test_sequences = tokenizer.texts_to_sequences(X_test)

# Padding the squences
padded_train_sequences = pad_sequences(train_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)
padded_test_sequences = pad_sequences(test_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

# Create Model

## Import library

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.layers import GlobalAveragePooling1D, BatchNormalization
from tensorflow.keras.layers import Flatten, GlobalMaxPool1D, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.optimizers import Adam, RMSprop
from tensorflow.keras.regularizers import l2, l1_l2

## Model A1

In [ ]:
# create model A1
model_A1 = Sequential()
model_A1.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length))
model_A1.add(GlobalAveragePooling1D())
model_A1.add(Dense(24, activation='relu'))
model_A1.add(Dropout(0.3))
model_A1.add(Dense(8, activation='relu'))
model_A1.add(Dropout(0.3))
model_A1.add(Dense(1, activation='sigmoid'))

model_A1.summary()

In [ ]:
# compile model
model_A1.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# set up early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# model training
history = model_A1.fit(padded_train_sequences, y_train,
                      epochs=30,
                      validation_data=(padded_test_sequences, y_test),
                      batch_size=64,
                      callbacks=[early_stopping])

In [ ]:
# training & validation loss values
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')

# training & validation accuracy values
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

### Evaluate and save model

In [ ]:
import pickle

# Evaluate the model
loss, accuracy = model_A1.evaluate(padded_test_sequences, y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

y_pred_prob = model_A1.predict(padded_test_sequences)
y_pred = (y_pred_prob > 0.5).astype(int)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title("Confusion Matrix")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Save the model with HDF5 format
model_A1.save('hoax_detection_A1.h5')

# Save the model with pickle format
with open('tokenizer_A1.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

### Convert to TFLite

In [ ]:
# Convert the model with Select TF ops
converter = tf.lite.TFLiteConverter.from_keras_model(model_A1)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

# Save the model
with open('hoax_detection_A1.tflite', 'wb') as f:
    f.write(tflite_model)

print('Model has been saved as TensorFlow Lite format.')

## Model A2

In [ ]:
# create model A2
model_A2 = Sequential()
model_A2.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length))
model_A2.add(GlobalAveragePooling1D())
model_A2.add(Dense(32, activation='relu'))
model_A2.add(Dropout(0.3))
model_A2.add(Dense(1, activation='sigmoid'))

model_A2.summary()

In [ ]:
# compile model
model_A2.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# set up early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# model training
history = model_A2.fit(padded_train_sequences, y_train,
                      epochs=20,
                      validation_data=(padded_test_sequences, y_test),
                      batch_size=128,
                      callbacks=[early_stopping])

In [ ]:
# training & validation loss values
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')

# training & validation accuracy values
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

### Evaluate and save model

In [ ]:
import pickle

# Evaluate the model
loss, accuracy = model_A2.evaluate(padded_test_sequences, y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

y_pred_prob = model_A2.predict(padded_test_sequences)
y_pred = (y_pred_prob > 0.5).astype(int)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title("Confusion Matrix")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Save the model with HDF5 format
model_A2.save('hoax_detection_A2.h5')

# Save the model with pickle format
with open('tokenizer_A2.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

### Convert to TFLite

In [ ]:
# Convert the model with Select TF ops
converter = tf.lite.TFLiteConverter.from_keras_model(model_A2)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

# Save the model
with open('hoax_detection_A2.tflite', 'wb') as f:
    f.write(tflite_model)

print('Model has been saved as TensorFlow Lite format.')

## Model A3

In [ ]:
# create model A3
model_A3 = Sequential()
model_A3.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length))
model_A3.add(GlobalAveragePooling1D())
model_A3.add(Dense(24, activation='relu'))
model_A3.add(BatchNormalization())
model_A3.add(Dropout(0.5))
model_A3.add(Dense(6, activation='relu'))
model_A3.add(BatchNormalization())
model_A3.add(Dropout(0.5))
model_A3.add(Dense(1, activation='sigmoid'))

model_A3.summary()

In [ ]:
# compile model
model_A3.compile(optimizer=RMSprop(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# set up early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# model training
history = model_A3.fit(padded_train_sequences, y_train,
                      epochs=20,
                      validation_data=(padded_test_sequences, y_test),
                      batch_size=64,
                      callbacks=[early_stopping])

In [ ]:
# training & validation loss values
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')

# training & validation accuracy values
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

### Evaluate and save model

In [ ]:
import pickle

# Evaluate the model
loss, accuracy = model_A3.evaluate(padded_test_sequences, y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

y_pred_prob = model_A3.predict(padded_test_sequences)
y_pred = (y_pred_prob > 0.5).astype(int)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title("Confusion Matrix")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Save the model with HDF5 format
model_A3.save('hoax_detection_A3.h5')

# Save the model with pickle format
with open('tokenizer_A3.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

### Convert to TFLite

In [ ]:
# Convert the model with Select TF ops
converter = tf.lite.TFLiteConverter.from_keras_model(model_A3)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

# Save the model
with open('hoax_detection_A3.tflite', 'wb') as f:
    f.write(tflite_model)

print('Model has been saved as TensorFlow Lite format.')

## Model B1

In [ ]:
# create model B1
model_B1 = Sequential()
model_B1.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length))
model_B1.add(Bidirectional(LSTM(24, return_sequences=False)))
model_B1.add(Dropout(0.2))
model_B1.add(Dense(1, activation='sigmoid'))

model_B1.summary()

In [ ]:
# compile model
model_B1.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# set up early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# model training
history = model_B1.fit(padded_train_sequences, y_train,
                      epochs=30,
                      validation_data=(padded_test_sequences, y_test),
                      batch_size=64,
                      callbacks=[early_stopping])

In [ ]:
# training & validation loss values
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')

# training & validation accuracy values
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

### Evaluate and save model

In [ ]:
import pickle

# Evaluate the model
loss, accuracy = model_B1.evaluate(padded_test_sequences, y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

y_pred_prob = model_B1.predict(padded_test_sequences)
y_pred = (y_pred_prob > 0.5).astype(int)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title("Confusion Matrix")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Save the model with HDF5 format
model_B1.save('hoax_detection_B1.h5')

# Save the model with pickle format
with open('tokenizer_B1.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

### Convert to TFLite

In [ ]:
# Convert the model with Select TF ops
converter = tf.lite.TFLiteConverter.from_keras_model(model_B1)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

# Save the model
with open('hoax_detection_B1.tflite', 'wb') as f:
    f.write(tflite_model)

print('Model has been saved as TensorFlow Lite format.')

## Model B2

In [ ]:
# create model B2
model_B2 = Sequential()
model_B2.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length))
model_B2.add(Bidirectional(LSTM(16, return_sequences=True, kernel_regularizer=l2(0.01))))
model_B2.add(Bidirectional(LSTM(8)))
model_B2.add(Dropout(0.5))
model_B2.add(Dense(1, activation='sigmoid'))

model_B2.summary()

In [ ]:
# compile model
model_B2.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# set up early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# model training
history = model_B2.fit(padded_train_sequences, y_train,
                      epochs=20,
                      validation_data=(padded_test_sequences, y_test),
                      batch_size=64,
                      callbacks=[early_stopping])

In [ ]:
# training & validation loss values
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')

# training & validation accuracy values
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

### Evaluate and save model

In [ ]:
import pickle

# Evaluate the model
loss, accuracy = model_B2.evaluate(padded_test_sequences, y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

y_pred_prob = model_B2.predict(padded_test_sequences)
y_pred = (y_pred_prob > 0.5).astype(int)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title("Confusion Matrix")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Save the model with HDF5 format
model_B2.save('hoax_detection_B2.h5')

# Save the model with pickle format
with open('tokenizer_B2.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

### Convert to TFLite

In [ ]:
# Convert the model with Select TF ops
converter = tf.lite.TFLiteConverter.from_keras_model(model_B2)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

# Save the model
with open('hoax_detection_B2.tflite', 'wb') as f:
    f.write(tflite_model)

print('Model has been saved as TensorFlow Lite format.')

## Model B3

In [ ]:
# create model B3
model_B3 = Sequential()
model_B3.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length))
model_B3.add(Bidirectional(LSTM(24, return_sequences=True, kernel_regularizer=l1_l2(l1=0.01, l2=0.01))))
model_B3.add(Bidirectional(LSTM(6)))
model_B3.add(Dropout(0.5))
model_B3.add(Dense(1, activation='sigmoid'))

model_B3.summary()

In [ ]:
# compile model
model_B3.compile(optimizer=RMSprop(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# set up early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# model training
history = model_B3.fit(padded_train_sequences, y_train,
                      epochs=20,
                      validation_data=(padded_test_sequences, y_test),
                      batch_size=64,
                      callbacks=[early_stopping])

In [ ]:
# training & validation loss values
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')

# training & validation accuracy values
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

### Evaluate and save model

In [ ]:
import pickle

# Evaluate the model
loss, accuracy = model_B3.evaluate(padded_test_sequences, y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

y_pred_prob = model_B3.predict(padded_test_sequences)
y_pred = (y_pred_prob > 0.5).astype(int)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title("Confusion Matrix")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Save the model with HDF5 format
model_B3.save('hoax_detection_B3.h5')

# Save the model with pickle format
with open('tokenizer_B3.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

### Convert to TFLite

In [ ]:
# Convert the model with Select TF ops
converter = tf.lite.TFLiteConverter.from_keras_model(model_B3)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

# Save the model
with open('hoax_detection_B3.tflite', 'wb') as f:
    f.write(tflite_model)

print('Model has been saved as TensorFlow Lite format.')

## Model C1

In [ ]:
# create model C1
model_C1 = Sequential()
model_C1.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length))
model_C1.add(LSTM(64, return_sequences=True))
model_C1.add(Dropout(0.3))
model_C1.add(LSTM(32))
model_C1.add(Dropout(0.3))
model_C1.add(Dense(24, activation='relu'))
model_C1.add(BatchNormalization())
model_C1.add(Dropout(0.3))
model_C1.add(Dense(8, activation='relu'))
model_C1.add(Dropout(0.3))
model_C1.add(Dense(1, activation='sigmoid'))

model_C1.summary()

In [ ]:
# compile model
model_C1.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# set up early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# model training
history = model_C1.fit(padded_train_sequences, y_train,
                      epochs=30,
                      validation_data=(padded_test_sequences, y_test),
                      batch_size=64,
                      callbacks=[early_stopping])

In [ ]:
# training & validation loss values
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')

# training & validation accuracy values
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

### Evaluate and save model

In [ ]:
import pickle

# Evaluate the model
loss, accuracy = model_C1.evaluate(padded_test_sequences, y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

y_pred_prob = model_C1.predict(padded_test_sequences)
y_pred = (y_pred_prob > 0.5).astype(int)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title("Confusion Matrix")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Save the model with HDF5 format
model_C1.save('hoax_detection_C1.h5')

# Save the model with pickle format
with open('tokenizer_C1.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

### Convert to TFLite

In [ ]:
# Convert the model with Select TF ops
converter = tf.lite.TFLiteConverter.from_keras_model(model_C1)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

# Save the model
with open('hoax_detection_C1.tflite', 'wb') as f:
    f.write(tflite_model)

print('Model has been saved as TensorFlow Lite format.')

## Model C2

In [ ]:
# create model C2
model_C2 = Sequential()
model_C2.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length))
model_C2.add(LSTM(32, return_sequences=True))
model_C2.add(Dropout(0.3))
model_C2.add(LSTM(16))
model_C2.add(Dropout(0.3))
model_C2.add(Dense(24, activation='relu'))
model_C2.add(BatchNormalization())
model_C2.add(Dropout(0.3))
model_C2.add(Dense(8, activation='relu'))
model_C2.add(Dropout(0.3))
model_C2.add(Dense(1, activation='sigmoid'))

model_C2.summary()

In [ ]:
# compile model
model_C2.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# set up early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# model training
history = model_C2.fit(padded_train_sequences, y_train,
                      epochs=30,
                      validation_data=(padded_test_sequences, y_test),
                      batch_size=64,
                      callbacks=[early_stopping])

In [ ]:
# training & validation loss values
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')

# training & validation accuracy values
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

### Evaluate and save model

In [ ]:
import pickle

# Evaluate the model
loss, accuracy = model_C2.evaluate(padded_test_sequences, y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

y_pred_prob = model_C2.predict(padded_test_sequences)
y_pred = (y_pred_prob > 0.5).astype(int)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title("Confusion Matrix")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Save the model with HDF5 format
model_C2.save('hoax_detection_C2.h5')

# Save the model with pickle format
with open('tokenizer_C2.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

### Convert to TFLite

In [ ]:
# Convert the model with Select TF ops
converter = tf.lite.TFLiteConverter.from_keras_model(model_C2)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

# Save the model
with open('hoax_detection_C2.tflite', 'wb') as f:
    f.write(tflite_model)

print('Model has been saved as TensorFlow Lite format.')

# Download model

In [ ]:
files.download("hoax_detection_A1.tflite")
files.download("hoax_detection_A1.h5")
files.download("hoax_detection_A2.tflite")
files.download("hoax_detection_A2.h5")
files.download("hoax_detection_A3.tflite")
files.download("hoax_detection_A3.h5")
files.download("hoax_detection_B1.tflite")
files.download("hoax_detection_B1.h5")
files.download("hoax_detection_B2.tflite")
files.download("hoax_detection_B2.h5")
files.download("hoax_detection_B3.tflite")
files.download("hoax_detection_B3.h5")
files.download("hoax_detection_C1.tflite")
files.download("hoax_detection_C1.h5")
files.download("hoax_detection_C2.tflite")
files.download("hoax_detection_C2.h5")
files.download("tokenizer_A1.pkl")
files.download("tokenizer_A2.pkl")
files.download("tokenizer_A3.pkl")
files.download("tokenizer_B1.pkl")
files.download("tokenizer_B2.pkl")
files.download("tokenizer_B3.pkl")
files.download("tokenizer_C1.pkl")
files.download("tokenizer_C2.pkl")

# Testing

In [ ]:
import pickle
import tensorflow as tf
from keras.preprocessing.sequence import pad_sequences

# Load tokenizer from pickle file
with open('tokenizer_C2.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

# Load TFLite model
interpreter = tf.lite.Interpreter(model_path='hoax_detection_C2.tflite')
interpreter.allocate_tensors()

# Get input and output tensor information
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# news you want to predict [A: 4, A2: 3, A3:3, A4: 3, B: 3, C: 4, C2: 4, C4: 4]
#news_text = ["Ketua Umum Partai Gerindra Prabowo Subianto mempromosikan produk sprei. Dalam video tersebut, Prabowo mengatakan akan membagikan ribuan sprei karena berhasil menang satu putaran dalam Pemilihan Presiden (Pilpres) 2024."]
#news_text = ["Dewan Kehormatan Penyelenggara Pemilu (DKPP) belum menjadwalkan sidang terhadap aduan soal dugaan perbuatan asusila oleh Ketua KPU RI Hasyim Asy'ari terhadap anggota Panitia Pemilihan Luar Negeri (PPLN) yang bertugas di Eropa."]
#news_text = ["Serangan Israel ini menewaskan puluhan warga, termasuk anak-anak. Meskipun sudah berkali-kali saya sampaikan, tapi saya ingin mengulang lagi bahwa Indonesia mengecam keras serangan Israel ke Rafah"]
#news_text = ["Penggalian Jalan Tol di daerah Jawa Timur mengeluarkan Minyak Mentah dengan potensi yang cukup besar"]
#news_text = ["Bendungan Sepaku Semoi di Kabupaten Penajam Paser Utara akan memasok air baku untuk IKN sebesar 2.000 liter per detik."]
#news_text = ["Presiden Joko Widodo atau Jokowi mengecam keras serangan Israel ke Kota Rafah, Gaza Selatan, Palestina. Serangan Israel ini menewaskan puluhan warga, termasuk anak-anak. Meskipun sudah berkali-kali saya sampaikan, tapi saya ingin mengulang lagi bahwa Indonesia mengecam keras serangan Israel ke Rafah kata Jokowi di Kota Dumai, Riau, Sabtu (1/6/2024). Dia meminta Israel untuk menataati perintah dari Mahkamah Internasional. Termasuk, menghentikan serangan ke Kota Rafah, Gaza, Palestina. Dan Israel mestinya memiliki kewajiban untuk mentaati mahkamah internasional, termasuk penghentian serangan ke Palestina jelasnya. "]
news_text = ["Dishub bekerja sama dengan Polri akan menggelar razia kendaraan yang mati pajak, bagi kendaraan yang telat membayar pajak selama 3 tahun atau lebih, kendaraan akan langsung ditahan. Pada pesan tersebut juga menyertakan jadwal razia yang akan dilakukan. Bagi kendaraan yang telat bayar pajak. Berdasarkan data, ada ratusan ribu motor dan mobil yang belum bayar pajak yang masih menggunakan pelat lama. Bagi kendaraan yang telat bayar pajak 3 tahun atau lebih akan langsung dikandangin."]

# Tokenization and padding of news
new_sequences = tokenizer.texts_to_sequences(news_text)
max_len = 100  # Make sure the maximum length matches the one used when training the model
new_padded = pad_sequences(new_sequences, maxlen=max_len)

# Convert input data to float32 type
new_padded = new_padded.astype('float32')

# Set the input tensor with compacted data
interpreter.set_tensor(input_details[0]['index'], new_padded)

# Run the interpreter to make predictions
interpreter.invoke()

# Get the prediction result from the output tensor
predictions_tflite = interpreter.get_tensor(output_details[0]['index'])

# Interpreting prediction results
predicted_labels_tflite = [1 if pred > 0.5 else 0 for pred in predictions_tflite]

# Show the prediction result
for text, pred, label in zip(news_text, predictions_tflite, predicted_labels_tflite):
    print(f'Text: {text}')
    print(f'Prediction: {pred[0]:.4f}')
    print(f'Predicted Label: {"Hoax" if label == 1 else "Not Hoax"}')